# Registry defaults and overrides

Use `FinstackConfig.extensions` to override versioned registry defaults explicitly. The embedded registries remain the fallback; overrides are opt-in and travel with the config object.


## Setup

This notebook uses the local registry JSON files as starting payloads, edits copies, and stores the edited payloads under the same extension keys Rust uses when loading defaults from `FinstackConfig`.


In [1]:
import json
from pathlib import Path
from pprint import pprint

from finstack.core.config import FinstackConfig


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "Cargo.toml").exists() and (candidate / "finstack/core/src").exists():
            return candidate
    raise RuntimeError("could not locate rfin repository root")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
REPO_ROOT


PosixPath('/Users/jeickmeier/Projects/rfin')

## Extension keys

Extension keys are versioned and namespaced. Use the exact key for the registry you want to replace.


In [2]:
EXTENSION_KEYS = {
    "credit_assumptions": "core.credit_assumptions.v1",
    "rating_scales": "core.rating_scales.v1",
    "margin_registry": "valuations.margin_registry.v1",
    "analytics_defaults": "analytics.defaults.v1",
    "monte_carlo_defaults": "monte_carlo.defaults.v1",
    "portfolio_liquidity_defaults": "portfolio.liquidity_defaults.v1",
    "calibration_defaults": "valuations.calibration_defaults.v1",
    "contract_specs": "valuations.contract_specs.v1",
    "structured_credit_assumptions": "valuations.structured_credit_assumptions.v1",
    "tba_assumptions": "valuations.tba_assumptions.v1",
    "ecl_policy": "statements_analytics.ecl_policy.v1",
}

for name, key in EXTENSION_KEYS.items():
    print(f"{name:34s} {key}")


credit_assumptions                 core.credit_assumptions.v1
rating_scales                      core.rating_scales.v1
margin_registry                    valuations.margin_registry.v1
analytics_defaults                 analytics.defaults.v1
monte_carlo_defaults               monte_carlo.defaults.v1
portfolio_liquidity_defaults       portfolio.liquidity_defaults.v1
calibration_defaults               valuations.calibration_defaults.v1
contract_specs                     valuations.contract_specs.v1
structured_credit_assumptions      valuations.structured_credit_assumptions.v1
tba_assumptions                    valuations.tba_assumptions.v1
ecl_policy                         statements_analytics.ecl_policy.v1


## Override Monte Carlo defaults

Registry overrides are full payload replacements. Start from the embedded JSON, edit the fields you govern, then attach the full edited payload to the config.


In [3]:
mc_defaults_path = REPO_ROOT / "finstack/monte_carlo/data/defaults/pricer_defaults.v1.json"
mc_defaults = json.loads(mc_defaults_path.read_text())

print("Embedded Python European-pricer defaults:")
pprint(mc_defaults["python_bindings"]["european_pricer"])
print("Embedded default currency:", mc_defaults["python_bindings"]["default_currency"])


Embedded Python European-pricer defaults:
{'num_paths': 100000, 'num_steps': 252, 'seed': 42, 'use_parallel': False}
Embedded default currency: USD


In [4]:
mc_override = json.loads(json.dumps(mc_defaults))
mc_override["python_bindings"]["european_pricer"]["num_paths"] = 25_000
mc_override["python_bindings"]["european_pricer"]["seed"] = 20260426
mc_override["python_bindings"]["default_currency"] = "EUR"

cfg = FinstackConfig()
cfg.set_extension(EXTENSION_KEYS["monte_carlo_defaults"], mc_override)

print("Configured extension keys:")
print(cfg.extension_keys())
print()
print("Overridden Python European-pricer defaults:")
pprint(cfg.get_extension(EXTENSION_KEYS["monte_carlo_defaults"])["python_bindings"]["european_pricer"])


Configured extension keys:
['monte_carlo.defaults.v1']

Overridden Python European-pricer defaults:
{'num_paths': 25000, 'num_steps': 252, 'seed': 20260426, 'use_parallel': False}


## Add analytics calendar defaults

The fiscal calendar defaults used by analytics can be governed the same way. Here we keep the exchange calendar explicit because the start day needs business-day alignment logic.


In [5]:
analytics_defaults_path = REPO_ROOT / "finstack/analytics/data/defaults/analytics_defaults.v1.json"
analytics_defaults = json.loads(analytics_defaults_path.read_text())

analytics_override = json.loads(json.dumps(analytics_defaults))
fiscal_calendar = analytics_override["python_bindings"]["lookback"]["default_fiscal_calendar"]
fiscal_calendar.update(
    {
        "id": "company_fiscal_april_nyse",
        "calendar_id": "nyse",
        "start_month": 4,
        "start_day": 1,
    }
)

cfg.set_extension(EXTENSION_KEYS["analytics_defaults"], analytics_override)

pprint(cfg.get_extension(EXTENSION_KEYS["analytics_defaults"])["python_bindings"]["lookback"])


{'default_fiscal_calendar': {'calendar_id': 'nyse',
                             'id': 'company_fiscal_april_nyse',
                             'start_day': 1,
                             'start_month': 4}}


## Serialize and round-trip

`FinstackConfig` can be serialized to JSON and restored. Use this payload when passing config into Python APIs that accept a config JSON string, or when handing config to Rust service code.


In [6]:
config_json = cfg.to_json()
print(config_json[:500] + "...")

round_tripped = FinstackConfig.from_json(config_json)
print(round_tripped.extension_keys())
print(round_tripped.get_extension(EXTENSION_KEYS["monte_carlo_defaults"])["python_bindings"]["default_currency"])


{"rounding":{"mode":"Bankers","ingest_scale":{"overrides":{}},"output_scale":{"overrides":{}}},"tolerances":{"rate_epsilon":1e-12,"generic_epsilon":1e-10},"extensions":{"analytics.defaults.v1":{"python_bindings":{"benchmark":{"ann_factor":252.0,"annualize":true,"rolling_window":63},"downside_deviation":{"ann_factor":252.0,"annualize":true,"mar":0.0},"lookback":{"default_fiscal_calendar":{"calendar_id":"nyse","id":"company_fiscal_april_nyse","start_day":1,"start_month":4}},"mean_return":{"ann_fac...
['analytics.defaults.v1', 'monte_carlo.defaults.v1']
EUR


## Remove an override

Because overrides are explicit, cleanup is just as explicit. Removing a key restores the embedded fallback for code that loads that registry from the config.


In [7]:
removed = cfg.remove_extension(EXTENSION_KEYS["analytics_defaults"])
print("removed analytics override:", removed)
print("remaining keys:", cfg.extension_keys())


removed analytics override: True
remaining keys: ['monte_carlo.defaults.v1']


## Practical rule

Keep overrides as reviewed JSON artifacts. Load the embedded registry payload, change only the governed fields, validate in CI, and attach the whole payload to `FinstackConfig`. Avoid partial fragments unless a specific registry documents overlay semantics; margin is the existing overlay-style exception.
